# Daily Challenge — Text Preprocessing & Analysis

Corpus: three Lewis Carroll books from Project Gutenberg
- *Alice's Adventures in Wonderland*
- *Through the Looking-Glass*
- *A Tangled Tale*

## Setup · Virtual environment & library installation

Create and activate a dedicated virtual environment **once** in your terminal before launching Jupyter:

```bash
python -m venv nlp_env
source nlp_env/bin/activate          # macOS / Linux
# nlp_env\Scripts\activate           # Windows
pip install nltk spacy wordcloud matplotlib scikit-learn requests
python -m spacy download en_core_web_sm
```

Then launch Jupyter from inside the activated environment.

In [ ]:
# Run once if working in Colab or without a virtual env
%pip install -q nltk spacy wordcloud matplotlib scikit-learn requests
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
import requests
import nltk
import spacy
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

# Download required NLTK resources
for resource in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger',
                 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker',
                 'maxent_ne_chunker_tab', 'words']:
    nltk.download(resource, quiet=True)

nlp = spacy.load('en_core_web_sm')
print('All libraries loaded.')

---
# Part 1 · Text Preprocessing

## Exercise 1 · `load_texts()` — fetch and clean raw corpus

In [ ]:
URLS = [
    'https://www.gutenberg.org/cache/epub/11/pg11.txt',    # Alice in Wonderland
    'https://www.gutenberg.org/cache/epub/12/pg12.txt',    # Through the Looking-Glass
    'https://www.gutenberg.org/cache/epub/29042/pg29042.txt',  # A Tangled Tale
]

TITLES = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale",
]


def load_texts(urls: list) -> list:
    """
    Fetch each URL, decode as UTF-8, strip non-word characters
    (keeping letters, digits, spaces, and newlines), and return
    a list of cleaned text strings.
    """
    corpus = []
    for url in urls:
        response = requests.get(url)
        response.encoding = 'utf-8'
        raw = response.text
        # Keep only word characters, whitespace, and apostrophes
        cleaned = re.sub(r"[^\w\s']", ' ', raw)
        # Collapse multiple spaces into one
        cleaned = re.sub(r' +', ' ', cleaned)
        corpus.append(cleaned)
    return corpus


corpus_raw = load_texts(URLS)
print(f"{len(corpus_raw)} texts loaded.")

## Exercise 2 · Preview and trim Gutenberg boilerplate

In [ ]:
# Print the first 200 characters of each text to spot the boilerplate
for title, text in zip(TITLES, corpus_raw):
    print(f"=== {title} ===")
    print(text[:200])
    print()

In [ ]:
def trim_gutenberg(text: str) -> str:
    """
    Remove Project Gutenberg header and footer by slicing between
    the ' START' and ' END' markers.
    """
    start_marker = ' START'
    end_marker   = ' END'

    start_idx = text.find(start_marker)
    end_idx   = text.find(end_marker)

    if start_idx != -1:
        text = text[start_idx + len(start_marker):]
    if end_idx != -1:
        # Recalculate end position relative to the already-sliced text
        end_idx = text.find(end_marker)
        if end_idx != -1:
            text = text[:end_idx]
    return text.strip()


corpus = [trim_gutenberg(t) for t in corpus_raw]

print("First 200 characters after trimming:")
for title, text in zip(TITLES, corpus):
    print(f"--- {title} ---")
    print(text[:200])
    print()

## Exercise 3 · Tokenization

In [ ]:
tokenized_corpus = [word_tokenize(text.lower()) for text in corpus]

for title, tokens in zip(TITLES, tokenized_corpus):
    print(f"=== {title} ===")
    print(f"Total tokens : {len(tokens)}")
    print(f"First 150    : {tokens[:150]}")
    print()

## Exercise 4 · Stopword removal

In [ ]:
stop_words = set(stopwords.words('english'))

# Remove stopwords and non-alphabetic tokens
filtered_corpus = [
    [token for token in tokens if token.isalpha() and token not in stop_words]
    for tokens in tokenized_corpus
]

# Verify removal: count common stopwords before and after
check_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']
print("Stopword count verification:")
print(f"{'Word':<12} {'Before':>8} {'After':>8}")
print("-" * 32)
for w in check_words:
    before = sum(t.count(w) for t in tokenized_corpus)
    after  = sum(t.count(w) for t in filtered_corpus)
    print(f"{w:<12} {before:>8} {after:>8}")

print()
for title, tokens in zip(TITLES, filtered_corpus):
    print(f"{title}: {len(tokens)} tokens after stopword removal")

## Exercise 5 · Stemming with PorterStemmer

In [ ]:
stemmer = PorterStemmer()

stemmed_corpus = [
    [stemmer.stem(token) for token in tokens]
    for tokens in filtered_corpus
]

for title, tokens in zip(TITLES, stemmed_corpus):
    print(f"=== {title} ===")
    print(tokens[:50])
    print()

## Exercise 6 · Lemmatization with spaCy

In [ ]:
lemmatized_corpus = []

for tokens in filtered_corpus:
    # Join filtered tokens back into a string for spaCy processing
    text_for_spacy = ' '.join(tokens)
    doc = nlp(text_for_spacy)
    lemmas = [token.lemma_ for token in doc if token.is_alpha()]
    lemmatized_corpus.append(lemmas)

for title, tokens in zip(TITLES, lemmatized_corpus):
    print(f"=== {title} ===")
    print(tokens[:50])
    print()

## Exercise 7 · Stemming vs Lemmatization — analysis

Let's compare the two side by side on the same tokens.

In [ ]:
N = 30
print(f"{'Original':<20} {'Stemmed':<20} {'Lemmatized':<20}")
print("-" * 60)
for orig, stem, lemma in zip(filtered_corpus[0][:N],
                              stemmed_corpus[0][:N],
                              lemmatized_corpus[0][:N]):
    print(f"{orig:<20} {stem:<20} {lemma:<20}")

### Analysis

| Aspect | Stemming (Porter) | Lemmatization (spaCy) |
|--------|-------------------|----------------------|
| **Approach** | Heuristic rule-based suffix stripping | Dictionary lookup + morphological analysis |
| **Output** | May not be a real word (e.g. `'curi'` from `'curious'`) | Always a valid base form (e.g. `'curious'`) |
| **Speed** | Very fast | Slower (requires a full NLP model) |
| **Accuracy** | Lower — oversimplifies | Higher — context-aware |
| **Use case** | Information retrieval at scale | NLP tasks requiring readable, correct words |

**Why the difference?**
Porter Stemmer blindly chops suffixes without checking whether the result is a real word
(e.g. `'running'` → `'run'` but `'universe'` → `'univers'`).
spaCy's lemmatizer uses the model's vocabulary and POS context to return the canonical
dictionary form, so the output is always interpretable.

## Exercise 8 · POS tagging with NLTK

In [ ]:
for title, tokens in zip(TITLES, filtered_corpus):
    pos_tags = nltk.pos_tag(tokens[:50])   # tag first 50 tokens for readability
    print(f"=== {title} (first 50 tokens) ===")
    print(pos_tags)
    print()

# Reminder of tag meanings
print("Tag legend examples:")
for tag in ['NN', 'NNS', 'VB', 'VBD', 'JJ', 'RB', 'PRP']:
    nltk.help.upenn_tagset(tag)

## Exercise 9 · Named Entity Recognition (NER) with NLTK

In [ ]:
def extract_entities(tokens):
    """Return all named entities found in a token list."""
    pos_tags = nltk.pos_tag(tokens)
    chunks   = nltk.ne_chunk(pos_tags, binary=False)
    entities = []
    for subtree in chunks:
        if hasattr(subtree, 'label'):
            entity_text  = ' '.join(w for w, _ in subtree.leaves())
            entity_label = subtree.label()
            entities.append((entity_text, entity_label))
    return entities


for title, tokens in zip(TITLES, filtered_corpus):
    entities = extract_entities(tokens[:500])  # limit for speed
    print(f"=== {title} ===")
    print(f"Entities found (in first 500 tokens): {entities[:20]}")
    print()

---
# Part 2 · Analysing the Text

## Exercise 1 · Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title, tokens in zip(axes, TITLES, lemmatized_corpus):
    text_for_cloud = ' '.join(tokens)
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        max_words=100,
        colormap='viridis'
    ).generate(text_for_cloud)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Clouds — Lewis Carroll Corpus', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Exercise 2 · Bag of Words — 5 most frequent words per book

In [ ]:
# Lemmatized text gives the cleanest BoW (no morphological noise)
bow_texts = [' '.join(tokens) for tokens in lemmatized_corpus]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(bow_texts)

feature_names = vectorizer.get_feature_names_out()

print("5 most frequent words per book (BoW):\n")
top5_bow = []
for i, title in enumerate(TITLES):
    counts = bow_matrix[i].toarray()[0]
    top_indices = counts.argsort()[-5:][::-1]
    top5 = [(feature_names[j], counts[j]) for j in top_indices]
    top5_bow.append(top5)
    print(f"{title}: {top5}")

## Exercise 3 · Inspect the BoW matrix

The BoW matrix has shape `(n_documents, n_vocabulary)`.  
- **Row index** = document number (0, 1, 2 here)  
- **Column index** = word index in the vocabulary  
- **Value** = number of times that word appears in that document

In [ ]:
print(f"BoW matrix shape : {bow_matrix.shape}")
print(f"  → {bow_matrix.shape[0]} documents, {bow_matrix.shape[1]} unique words\n")

# Show a few (word, index, count) triplets for document 0
doc_idx = 0
counts  = bow_matrix[doc_idx].toarray()[0]
nonzero = np.nonzero(counts)[0][:15]  # first 15 non-zero entries

print(f"Sample entries for document {doc_idx} — '{TITLES[doc_idx]}':")
print(f"{'Word':<20} {'Vocab index':>12} {'Count':>8}")
print("-" * 44)
for idx in nonzero:
    print(f"{feature_names[idx]:<20} {idx:>12} {int(counts[idx]):>8}")

## Exercise 4 · Pie charts — 5 most frequent words (BoW)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, title, top5 in zip(axes, TITLES, top5_bow):
    words  = [w for w, _ in top5]
    counts = [c for _, c in top5]
    labels = [f"{w}\n({c})" for w, c in top5]
    ax.pie(
        counts, labels=labels,
        autopct='%1.1f%%',
        startangle=140,
        colors=plt.cm.Set2.colors[:5]
    )
    ax.set_title(f"{title}\n(BoW top 5)", fontsize=10, fontweight='bold')

plt.suptitle('5 Most Frequent Words — Bag of Words', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Exercise 5 · Analysis of BoW results

**Are those words informative?**

Not very. Words like *alice*, *say*, *come*, *go*, *little* dominate the frequency charts
because they appear constantly throughout all three books. They are **expected** words:

- *alice* is the protagonist — obviously the most mentioned word in the Alice books.
- *say*, *come*, *go*, *little* are extremely common narrative verbs/adjectives in 19th-century prose.

**The core problem with BoW**: it treats all documents equally, so ubiquitous words across
the corpus drown out words that might be truly distinctive to a specific book.
→ This motivates the use of **TF-IDF** in the next section.

---
# Part 3 · Solving the Frequency Problem with TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) down-weights words that are
common across all documents (low IDF) and up-weights words that are frequent in one
document but rare in others (high IDF). This surfaces the words that truly characterise
each specific book.

## Exercise 1 · TF-IDF vectorizer

In [ ]:
# min_df=1, max_df=2 are required for a tiny 3-document corpus:
# - min_df=1 : include words that appear in at least 1 document
# - max_df=2 : exclude words that appear in more than 2 documents
#              (i.e., words present in ALL 3 books are filtered out as too generic)
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix     = tfidf_vectorizer.fit_transform(bow_texts)

tfidf_features = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print()

print("5 most relevant words per book (TF-IDF):\n")
top5_tfidf = []
for i, title in enumerate(TITLES):
    scores      = tfidf_matrix[i].toarray()[0]
    top_indices = scores.argsort()[-5:][::-1]
    top5        = [(tfidf_features[j], round(float(scores[j]), 4)) for j in top_indices]
    top5_tfidf.append(top5)
    print(f"{title}: {top5}")

## Exercise 2 · Pie charts — 5 most relevant words (TF-IDF)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, title, top5 in zip(axes, TITLES, top5_tfidf):
    words  = [w for w, _ in top5]
    scores = [s for _, s in top5]
    labels = [f"{w}\n({s:.4f})" for w, s in top5]
    ax.pie(
        scores, labels=labels,
        autopct='%1.1f%%',
        startangle=140,
        colors=plt.cm.Set3.colors[:5]
    )
    ax.set_title(f"{title}\n(TF-IDF top 5)", fontsize=10, fontweight='bold')

plt.suptitle('5 Most Relevant Words — TF-IDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Final Analysis · BoW vs TF-IDF

| | BoW | TF-IDF |
|---|---|---|
| Top words | Generic, shared across books (*alice*, *say*, *go*) | Book-specific, distinctive terms |
| Informativeness | Low — expected words everywhere | High — reveals what makes each book unique |
| Best for | Baseline frequency counting | Document characterisation, retrieval |

**Key insight**: TF-IDF removes the noise introduced by words that dominate every document.
The resulting top words are far more characteristic of each individual book,
making them genuinely useful for tasks like classification or retrieval.